# Machine-learning implementation plan

This notebook is the decision record and implementation roadmap for improving the first synthetic Einstein-radius model. It does not train a model itself and it does not authorize using real observations as labelled training data.

**Primary target:** `theta_E_arcsec`, the simulated Einstein-radius parameter in arcseconds.

**Primary input:** a `256 x 256` processed simulated image. Future input experiments may add masks, variance, and PSF information only when each product would also be available in the intended deployment domain.

## Current baseline: recorded, not a tuning target

The first baseline is a 248,409-parameter four-block CNN trained on `sim-eng1000-v1`. It uses a Huber loss and AdamW, computes normalization from the training split only, and selects its checkpoint by validation MAE.

| Split | Parents | Role |
|---|---:|---|
| Train | 700 | fit weights and normalization |
| Validation | 150 | choose architecture, loss, inputs, and checkpoint |
| Test | 150 | recorded baseline report only |

Recorded epoch-19 results:

| Metric | Validation | Test |
|---|---:|---:|
| MAE | 0.0884 arcsec | 0.0876 arcsec |
| RMSE | 0.1163 arcsec | 0.1292 arcsec |
| Median relative error | 5.71% | 4.93% |
| Bias | -0.0031 arcsec | -0.0090 arcsec |
| R2 | 0.9285 | 0.9127 |

The test MAE is 77.2% lower than the train-mean baseline. This result applies only to held-out images from the same synthetic engineering proposal.

## Non-negotiable evaluation rules

1. Do not select future architectures, losses, inputs, epochs, or augmentations from the existing test metrics. Use train and validation only.
2. Preserve `sim-eng1000-v1` test results as the baseline record.
3. Generate an independently seeded final holdout release before making a final challenger-model claim.
4. Keep `sim-p100-v1` as a separate stress/out-of-distribution suite; do not silently merge it into training.
5. Record the config, seed, release hash, split hashes, package versions, checkpoint hash, and all metrics for every run.
6. Never provide targets, seeds, source/lens truth, ancestry, IDs, or truth-sidecar paths to a model input.

## Phase 1: establish a stable development benchmark

Run the current baseline with three fixed seeds: `12001`, `12002`, and `12003`. Keep all other settings fixed.

For each seed, record validation MAE, RMSE, median relative error, bias, R2, 90th percentile absolute error, 95th percentile absolute error, best epoch, and runtime.

**Gate:** do not rank small improvements unless the validation improvement is larger than seed-to-seed variation. Select candidates by mean validation MAE, while rejecting candidates with materially worse tail error or radius-dependent bias.

In [ ]:
# Illustrative command pattern; create one versioned config per seed.
# python -m src.ml.train --config configs/ml/seed-12001.yaml
# python -m src.ml.train --config configs/ml/seed-12002.yaml
# python -m src.ml.train --config configs/ml/seed-12003.yaml
# Do not run the existing test split while selecting among these runs.

## Phase 2: input ablations

Hold the current CNN, Huber loss, training schedule, and seed protocol fixed. Change exactly one input representation at a time.

| ID | Input channels | Question |
|---|---|---|
| IN-00 | processed image | Current reference |
| IN-01 | image + invalid/coverage masks | Do explicit valid-pixel boundaries help? |
| IN-02 | image + variance | Does declared pixel uncertainty help? |
| IN-03 | image + masks + variance | Do the products complement each other? |
| IN-04 | training-only asinh or robust intensity transform | Does dynamic-range compression help faint structures? |
| IN-05 | image conditioned on PSF kernel | Does explicit blur information help robustly? |

Do not use simulator-only scalar metadata such as the generating PSF FWHM as a feature. A PSF input is acceptable only when represented by a product that could exist for real deployment. Assess every input option by overall and subgroup validation metrics, especially low-radius doubles and weak detectable-image configurations.

## Phase 3: loss-function ablations

Use the same model, inputs, seeds, epoch budget, and checkpoint-selection rule.

| ID | Loss | Expected tradeoff |
|---|---|---|
| LO-00 | Huber | Current robust reference |
| LO-01 | MSE | Prioritizes large failures; may improve outliers while harming typical error |
| LO-02 | MAE | Prioritizes typical error; may tolerate catastrophic errors |
| LO-03 | log-cosh | Smooth robust alternative |

Report MAE, RMSE, median relative error, bias, and the 90th/95th percentile absolute errors. Do not choose a loss solely because it improves a single headline metric. Weighted losses are deferred until a scientifically justified weighting policy exists; weak systems must remain visible in reporting.

## Phase 4: architecture ablations

Increase complexity only after the input and loss experiments establish a reliable comparison protocol.

1. Current compact CNN.
2. Wider/deeper CNN with a controlled parameter-count increase.
3. Small residual CNN.
4. CNN or residual CNN with coordinate channels so spatial geometry is explicit.
5. PSF/mask/variance-aware residual model, if Phase 2 supports those inputs.
6. Multi-task model for `theta_E_arcsec`, `q`, `phi_rad`, and ellipticity only after scalar radius inference is stable.

Avoid large pretrained vision models at this stage. There are only 700 training parents, so a modest model with transparent ablations is more informative than a high-capacity model that may memorize simulator artifacts.

## Phase 5: robustness and failure analysis

The current model has two large low-radius-double errors. Treat this as a hypothesis, not a conclusion: weak or ambiguous configurations may be intrinsically difficult.

Run controlled validation studies that vary one factor at a time:

- PSF width and mismatch;
- Poisson/read-noise level;
- source brightness, morphology, and position;
- lens-light brightness;
- image detectability and missing/weak counterimages;
- target radius bin;
- physical versus detectable image multiplicity;
- P100 shear/source-complexity stress cases.

Create a robustness matrix with MAE, relative error, tail error, and bias for each condition. The goal is to identify where the model is reliable and where it is not, not to hide hard cases.

## Phase 6: uncertainty

The current CNN returns one value with no uncertainty. Add uncertainty only after the deterministic model and robustness report are stable.

Candidate methods:

- Deep ensemble: train 3 to 5 fixed-seed copies and use prediction spread as an epistemic uncertainty proxy.
- Heteroscedastic regression: predict mean and observation-scale uncertainty with Gaussian negative log likelihood.
- Quantile regression: predict 16th, 50th, and 84th percentiles.

Evaluate nominal 68% and 95% coverage, interval width, negative log likelihood where applicable, calibration curves, and whether predicted uncertainty rises for actual large errors. These are engineering uncertainty diagnostics, not astrophysical posterior claims.

## Phase 7: final challenger evaluation

Before a final result, freeze one candidate's architecture, inputs, loss, augmentation, seed policy, training budget, checkpoint rule, metrics, and subgroup reports.

Then generate a new independently seeded synthetic holdout release and evaluate exactly once. Compare the challenger with:

- train-mean baseline;
- current CNN baseline;
- classical recovery grid;
- P100 stress performance;
- uncertainty calibration, if implemented.

The final model card must state training data, target definition, inputs, restrictions, overall metrics, subgroup/tail results, robustness outcomes, calibration, compute, and known failure modes.

## Phase 8: bounded real-data exploration

Real-data use comes after synthetic robustness, not before. Run the frozen model only on the preselected exploratory systems, compare an angular scale with compatible references, show the image plus prediction and uncertainty, and state the simulation-to-real limitations.

Do not present these outputs as physical mass measurements, calibrated posteriors, population results, or validated real-HST accuracy. Empirical PSFs, detector/error products, source morphology, calibrated priors, and blind real benchmarks remain required for those claims.

## Experiment-record template

Every experiment should write a versioned YAML/JSON config and a report with:

```text
experiment_id
parent_baseline_id
release_id + manifest hash + split hashes
input channels and preprocessing
model architecture + trainable parameter count
loss, optimizer, schedule, epochs, batch size, seeds
checkpoint-selection metric and epoch
overall validation metrics
subgroup/tail/robustness metrics
plots, predictions, runtime, hardware, package versions
scientific scope and limitations
```

Use a new output directory for every experiment. Never overwrite an accepted experiment artifact with an incompatible configuration.

## Immediate implementation backlog

- [ ] Add an experiment registry and versioned configuration layout.
- [ ] Add a multi-seed training runner and aggregate validation report.
- [ ] Add P90/P95 error and target-bin metrics.
- [ ] Implement mask/variance input-channel adapters.
- [ ] Run Huber, MSE, MAE, and log-cosh loss ablations.
- [ ] Implement a small residual CNN.
- [ ] Add P100 and controlled robustness evaluations.
- [ ] Add ensemble uncertainty and calibration diagnostics.
- [ ] Generate an independent final holdout release.
- [ ] Freeze one challenger and write a model card.
- [ ] Perform the bounded real-data exploratory study.

The immediate next implementation task is the multi-seed validation runner. It quantifies run-to-run variation before changes to models, losses, or inputs are interpreted as improvements.